In [ ]:
import os
from databricks.connect import DatabricksSession
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pyspark.sql.functions import col
from sentence_transformers import SentenceTransformer
from databricks.ai_search.client import VectorSearchClient

CATALOG = "financial_db"
SCHEMA = "filings"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.sec_rag_silver"
TABLE = f"{CATALOG}.{SCHEMA}.sec_rag_gold"

c:\Users\migue\Documents\Projects\sec_sentiment_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ.pop("DATABRICKS_METADATA_SERVICE_URL", None)
os.environ.pop("DATABRICKS_AUTH_TYPE", None)

spark = DatabricksSession.builder.getOrCreate()

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TABLE} (
    chunk_id STRING,
    ticker STRING,
    cik STRING,
    date STRING,
    accession_number STRING,
    url STRING,
    chunk_index INT,
    chunk_text STRING,
    chunk_length INT,
    sentiment_label STRING,
    sentiment_pos FLOAT,
    sentiment_neg FLOAT,
    sentiment_neu FLOAT,
    embedding ARRAY<FLOAT>
)
USING DELTA TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
);
''')

""


In [3]:
def process_sentiment(text_series):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "ProsusAI/finbert"
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()
    
    labels = ["positive", "negative", "neutral"]
    results = []
    
    with torch.no_grad():
        texts = text_series.tolist()
        inputs = tokenizer(
            texts, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors="pt"
        ).to(device)
        
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()
        
        for prob in probs:
            pos, neg, neu = float(prob[0]), float(prob[1]), float(prob[2])
            top_label = labels[prob.argmax()]
            results.append({
                "label": top_label,
                "pos": pos,
                "neg": neg,
                "neu": neu
            })
            
    return results

def process_embeddings(text_series):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "BAAI/bge-small-en-v1.5"
    
    model = SentenceTransformer(model_name, device=device)
    texts = text_series.tolist()
    
    embeddings = model.encode(texts, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    
    return [emp.tolist() for emp in embeddings]

def get_gold():
    df_silver = spark.sql(f"""
        SELECT chunk_id, ticker, cik, date, accession_number, url, 
            chunk_index, chunk_text, chunk_length
        FROM {SILVER_TABLE}
        WHERE chunk_id NOT IN (SELECT DISTINCT chunk_id FROM {TABLE})
    """)

    if df_silver.count() > 0:
        df_pd = df_silver.toPandas()
        df_pd["sentiment"] = process_sentiment(df_pd["chunk_text"])
        df_pd["embedding"] = process_embeddings(df_pd["chunk_text"])
        df = spark.createDataFrame(df_pd)
        df = df.select(
                col("chunk_id"),
                col("ticker"),
                col("cik"),
                col("date"),
                col("accession_number"),
                col("url"),
                col("chunk_index"),
                col("chunk_text"),
                col("chunk_length"),
                col("sentiment.label").alias("sentiment_label"),
                col("sentiment.pos").alias("sentiment_pos"),
                col("sentiment.neg").alias("sentiment_neg"),
                col("sentiment.neu").alias("sentiment_neu"),
                col("embedding")
            )
        df.write.format("delta").mode("append").saveAsTable(TABLE)

get_gold()

In [1]:
vsc = VectorSearchClient()

endpoint_name = "financial_rag_endpoint"

existing_endpoints = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]

if endpoint_name not in existing_endpoints:
    vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")

index_name = f"{CATALOG}.{SCHEMA}.sec_gold_vector_index"

vsc.create_delta_sync_index(
    endpoint_name=endpoint_name,
    source_table_name=TABLE,
    index_name=index_name,
    pipeline_type="TRIGGERED",  
    primary_key="chunk_id",
    embedding_dimension=384,
    embedding_vector_column="embedding"
)

NameError: name 'VectorSearchClient' is not defined

In [7]:
spark.sql(f"""
    SELECT ticker, date, chunk_index, sentiment_label, 
           round(sentiment_pos, 2) as pos, round(sentiment_neg, 2) as neg,
           size(embedding) as vector_dim, substring(chunk_text, 1, 80) as text_snippet
    FROM {TABLE}
    ORDER BY sentiment_pos DESC
    LIMIT 10
""").toPandas()

,ticker,date,chunk_index,sentiment_label,pos,neg,vector_dim,text_snippet
0,TSLA,2026-01-29,1,positive,0.74,0.05,384,oducts. There is no guarantee we will be able ...
1,GOOGL,2026-02-05,0,positive,0.70,0.01,384,of this Annual Report on Form 10-K. Culture an...
2,TSLA,2026-01-29,23,positive,0.69,0.02,384,f the uptake rate for autonomous driving solut...
3,NVDA,2026-02-25,26,positive,0.66,0.02,384,to market; • competing technologies and compet...
4,TSLA,2026-01-29,14,positive,0.64,0.03,384,"eover, because of our unique expertise with ou..."
5,NVDA,2026-02-25,108,positive,0.63,0.04,384,"ong Kong and Macau) and D:5 countries, or to c..."
6,MSFT,2026-07-29,56,positive,0.63,0.01,384,"programs and initiatives, grants, and LinkedIn..."
7,GOOGL,2026-02-05,15,positive,0.60,0.01,384,ption of technological advancements by competi...
8,TSLA,2026-01-29,13,positive,0.54,0.18,384,"ve independent dealer networks, we are respons..."
9,AMZN,2026-02-06,59,positive,0.53,0.01,384,"ers, and/or greater brand recognition, particu..."
